# Flight Delay Prediction: Baseline Model

Working notebook for the [Zindi Flight Delay Prediction Challenge](https://zindi.africa/competitions/flight-delay-prediction-challenge) (Tunisair). This notebook builds Milestone 1: the **baseline model**, following the plan in `Flight_Delay_Baseline_and_Strategy_Report.pdf`.

## Define the Business Goal

- **Stakeholder:** Tunisair operations / the Zindi organisers evaluating submissions.
- **Prediction:** the delay of a given Tunisair flight, **in minutes** (can be negative for an early departure/arrival).
- **Problem type:** regression -- the target is a continuous number, not a delayed/not-delayed label.
- **Evaluation metric: Root Mean Squared Error (RMSE).** RMSE squares each error before averaging, so a few very large misses hurt the score far more than many small ones. This favours models that avoid big outlier misses over models that are only accurate on average, and it is also the metric the Zindi leaderboard scores submissions on.
- **Baseline model:** the simplest rule that makes a prediction. Three are built below -- a **naive mean/median predictor** (the absolute floor), a **route group-mean predictor**, and a **Ridge regression** on basic engineered features (the first "real" ML baseline). Every later, more complex model (Random Forest, XGBoost, ...) must beat these to justify its added complexity.

## Get the Data

> [!IMPORTANT]
> We import cleaned data from './data_cleaning.ipynb' (`data/cleaned_data.csv`). That file doesn't keep the flight date, so we also pull `DATOP` directly from the raw `data/Train.csv` to support a chronological train/test split.

In [8]:
import pandas as pd

data = pd.read_csv("data/cleaned_data.csv")

# cleaned_data.csv doesn't keep the flight date (only weekday/month/year), so pull it
# back in from the raw export to support a chronological split below.
dates = pd.read_csv("data/Train.csv", usecols=["ID", "DATOP"])
dates["id"] = dates["ID"].str.extract(r"(\d+)$").astype(int)
dates["DATOP"] = pd.to_datetime(dates["DATOP"])
data = data.merge(dates[["id", "DATOP"]], on="id")

data.head()

/var/folders/60/p84z97d51rz4wdn4800x9x1r0000gn/T/ipykernel_49819/4123133565.py:3: DtypeWarning: Columns (0: fltid_number) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("data/cleaned_data.csv")


,id,depstn,arrstn,target,scheduled_week_day_of_departure,scheduled_month_of_departure,scheduled_year_of_departure,scheduled_hour_of_departure,scheduled_minutes_of_departure,scheduled_week_day_of_arrival,scheduled_month_of_arrival,scheduled_year_of_arrival,scheduled_hour_of_arrival,scheduled_minutes_of_arrival,fltid_airline,fltid_number,ac_type,ac_tail,DATOP
0,0,CMN,TUN,260.0,Sunday,January,2016,10,30,Sunday,January,2016,12,55,TU,0712,32A,IMN,2016-01-03
1,1,MXP,TUN,20.0,Wednesday,January,2016,15,5,Wednesday,January,2016,16,55,TU,0757,31B,IMO,2016-01-13
2,2,TUN,IST,0.0,Saturday,January,2016,4,10,Saturday,January,2016,6,45,TU,0214,32A,IMN,2016-01-16
3,3,DJE,NTE,0.0,Sunday,January,2016,14,10,Sunday,January,2016,17,0,TU,0480,736,IOK,2016-01-17
4,4,TUN,ALG,22.0,Sunday,January,2016,14,30,Sunday,January,2016,15,50,TU,0338,320,IMU,2016-01-17


## Time-based Train Test Split

In [ ]:
data = data.sort_values("DATOP")

X = data.drop(columns=["target", "DATOP"])
y = data["target"]

# Time-based split, not random: earlier flights train, the most recent ~20% of dates
# are held out for testing. Needed because later features (historical average delay per
# route/aircraft/flight number, same-day delay-propagation) are computed over time -- a
# random split would let "future" rows leak into a "past" row's history.
cutoff = data["DATOP"].quantile(0.8)
train_mask = data["DATOP"] <= cutoff

X_train, X_test = X[train_mask], X[~train_mask]
y_train, y_test = y[train_mask], y[~train_mask]

print(f"Train: {len(X_train)} rows up to {data.loc[train_mask, 'DATOP'].max().date()}")
print(f"Test: {len(X_test)} rows from {data.loc[~train_mask, 'DATOP'].min().date()} onward")

Train: 86352 rows up to 2018-06-01
Test: 21481 rows from 2018-06-02 onward


## Baseline Models

Before training a real model, we set the RMSE and MAE floor with three baselines of increasing sophistication

In [10]:
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
import numpy as np 

### Baseline 0 -- Constant Predictor

Ignore every feature; always predict the training set's median and mean delay. If a later model cannot beat this, it has learned nothing useful.

In [11]:
baseline0_pred_mean = y_train.mean()
baseline0_pred_median = y_train.median()
y_pred_median = np.full(shape=y_test.shape, fill_value=baseline0_pred_median)
y_pred_mean = np.full(shape=y_test.shape, fill_value=baseline0_pred_mean)

median_mae = mean_absolute_error(y_test, y_pred_median)
median_rmse = root_mean_squared_error(y_test, y_pred_median)

mean_mae = mean_absolute_error(y_test, y_pred_mean)
mean_rmse = root_mean_squared_error(y_test, y_pred_mean)

print(f"Median baseline — MAE: {median_mae:.2f} min, RMSE: {median_rmse:.2f} min")
print(f"Mean baseline — MAE: {mean_mae:.2f} min, RMSE: {mean_rmse:.2f} min")

Median baseline — MAE: 60.48 min, RMSE: 150.14 min
Mean baseline — MAE: 65.59 min, RMSE: 142.10 min


### Baseline 1 -- Group-Mean Predictor

Predict the historical mean delay of the flight's **route** (`depstn`-`arrstn`), computed on training data only. No modelling required, but it already captures "this route is chronically late" signal.

In [12]:
route_mean = (
    X_train.assign(route=X_train["depstn"].astype(str) + "_" + X_train["arrstn"].astype(str), target=y_train)
    .groupby("route")["target"].mean()
)
global_mean = y_train.mean()  # fallback for routes not seen in training

test_route = X_test["depstn"].astype(str) + "_" + X_test["arrstn"].astype(str)
baseline1_pred = test_route.map(route_mean).fillna(global_mean)

rmse_baseline1 = root_mean_squared_error(y_test, baseline1_pred)
mae_baseline1 = mean_absolute_error(y_test, baseline1_pred)

print(f"Baseline 1 (route mean) validation RMSE: {rmse_baseline1:.2f} min, MAE: {mae_baseline1:.2f} min")

Baseline 1 (route mean) validation RMSE: 139.60 min, MAE: 61.87 min


### Baseline 2 -- Ridge Regression

A simple linear model over departure/arrival station, day of week and month of departure, and scheduled departure/arrival time (hour and minute). This is the first baseline that is a real (if simple) ML model, and the reference point for whether tree ensembles (Random Forest, XGBoost -- the next iteration) are actually earning their extra complexity.

In [13]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


cat_cols = ["depstn", "arrstn", "scheduled_week_day_of_departure", "scheduled_month_of_departure"]
num_cols = ["scheduled_hour_of_departure", "scheduled_minutes_of_departure", "scheduled_hour_of_arrival", "scheduled_minutes_of_arrival"]

preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(drop="first",handle_unknown="ignore"), cat_cols),
], remainder="passthrough")

baseline2 = Pipeline([("preprocess", preprocess), ("reg", Ridge(alpha=1.0))])
baseline2.fit(X_train[cat_cols + num_cols], y_train)

baseline2_pred = baseline2.predict(X_test[cat_cols + num_cols])
rmse_baseline2 = root_mean_squared_error(y_test, baseline2_pred)
mae_baseline2 = mean_absolute_error(y_test, baseline2_pred)

print(f"Baseline 2 (Ridge regression) validation RMSE: {rmse_baseline2:.2f}")
print(f"Baseline 2 (Ridge regression) validation MAE: {mae_baseline2:.2f}")

Baseline 2 (Ridge regression) validation RMSE: 139.48
Baseline 2 (Ridge regression) validation MAE: 60.67


/Users/lucatomarelli/spiced/projects/ml_project_flight_delay_prediction/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


### Compare the Baselines

Collects every baseline's score into one table, sorted by RMSE (the competition's metric), so it's clear at a glance which one currently sets the floor later models have to beat.

- **`validation_rmse` (Root Mean Squared Error, minutes):** squares each error before averaging, then takes the square root. Squaring punishes large misses far more than small ones, so RMSE is dominated by the handful of extreme delays in this data rather than by how well the model does on a typical flight. This is the leaderboard metric.
- **`validation_mae` (Mean Absolute Error, minutes):** averages the absolute size of each error, so a 10-minute miss and a 1000-minute miss count in proportion to their size, not squared. This reflects the "typical" error a flight sees and is far less sensitive to outliers than RMSE.
- The two metrics can disagree on which baseline "wins": the constant **median** predictor has the best MAE here but the worst RMSE, while the constant **mean** predictor is the opposite. That's not a bug -- the median is the constant that minimizes MAE by construction, and the mean is the constant that minimizes RMSE by construction. Reporting both is what surfaces this trade-off instead of hiding it.

In [14]:
pd.DataFrame({
    "baseline": ["Constant mean", "Constant median", "Route mean", "Ridge regression"],
    "validation_rmse": [mean_rmse, median_rmse, rmse_baseline1, rmse_baseline2],
    "validation_mae": [mean_mae, median_mae, mae_baseline1, mae_baseline2],
}).sort_values("validation_rmse")

,baseline,validation_rmse,validation_mae
3,Ridge regression,139.475343,60.666570
2,Route mean,139.597900,61.869353
0,Constant mean,142.102752,65.589091
1,Constant median,150.142592,60.482473


## References & Further Reading

- [**Flight Delay Prediction Challenge (Zindi)**](https://zindi.africa/competitions/flight-delay-prediction-challenge): the competition this notebook targets.
- `Flight_Delay_Baseline_and_Strategy_Report.pdf`: the fuller strategy report this notebook implements (data caveats, feature ideas, model comparison, validation pitfalls).
- [**root_mean_squared_error (scikit-learn)**](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.root_mean_squared_error.html): the evaluation metric used here and by the leaderboard.
- [**Ridge (scikit-learn)**](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html): the model behind Baseline 2.
- [**Common pitfalls and recommended practices (scikit-learn)**](https://scikit-learn.org/stable/common_pitfalls.html): general guidance on data leakage, directly relevant to the historical-average features planned for the next iteration.